# P09: Applications and advanced tips

`numpy` is the engine under `pandas`, `scipy`, and essentially every neural-data
toolbox. The recurring theme of this companion is *axes and shapes*: the same line
of code can compute completely different science depending on which axis it runs
over, and it never tells you when you have chosen the wrong one.

## The modern random API

In [1]:
import numpy as np

# the current best practice: make a Generator with an explicit seed
rng = np.random.default_rng(42)     # replaces the old np.random.seed(...) global state

# a (trials x timepoints) array of a recorded signal
data = rng.normal(0, 1, size=(50, 200))
print(data.shape)

(50, 200)


## Z-scoring: which axis?

In [2]:
# z-score each TIMEPOINT across TRIALS  ->  axis=0
z = (data - data.mean(axis=0)) / data.std(axis=0, ddof=1)
print("z shape:", z.shape, " grand mean ~", round(float(z.mean()), 3))

z shape: (50, 200)  grand mean ~ -0.0


:::{admonition} Double check: axis is a scientific choice, not a detail
:class: warning
`data.mean(axis=0)` averages over trials (giving one value per timepoint);
`data.mean(axis=1)` averages over time (one value per trial). Both run instantly
and return sensible-looking arrays of the *wrong* thing if you picked the wrong
axis. Always check `.shape` before and after, and write a comment stating what
each axis means. AI-generated array code frequently guesses the axis.
:::

## Baseline subtraction with broadcasting

In [3]:
# subtract each trial's OWN pre-stimulus baseline (first 50 samples)
baseline = data[:, :50].mean(axis=1, keepdims=True)   # keepdims -> shape (50, 1)
corrected = data - baseline                            # broadcasts across time
print("baseline:", baseline.shape, " corrected:", corrected.shape)

baseline: (50, 1)  corrected: (50, 200)


In [4]:
# without keepdims the baseline is shape (50,), which does NOT line up with (50, 200)
bad_shape = data[:, :50].mean(axis=1)
print("bad_shape:", bad_shape.shape)
# 'data - bad_shape' would raise; 'data - bad_shape[:, None]' is the explicit fix

bad_shape: (50,)


:::{admonition} Double check: broadcasting can silently do the wrong thing
:class: warning
NumPy's broadcasting is powerful and quiet. Mismatched-but-broadcastable shapes
-- e.g. a `(200,)` row vector where you meant a `(50, 1)` column vector -- can
subtract the wrong quantity from every element without any error. Use
`keepdims=True`, add explicit axes with `[:, None]`, and assert shapes
(`assert baseline.shape == (50, 1)`) when it matters.
:::

## Missing values poison ordinary reductions

In [5]:
x = np.array([1.0, 2.0, np.nan, 4.0])
print("np.mean:   ", np.mean(x))      # nan -- a single missing value spreads
print("np.nanmean:", np.nanmean(x))   # 2.333 -- skips the nan

np.mean:    nan
np.nanmean: 2.3333333333333335


:::{admonition} Double check: `np.mean` vs `np.nanmean` (and `np.corrcoef` with NaNs)
:class: warning
A single `NaN` turns the ordinary `np.mean`, `np.std`, and `np.corrcoef` results
into `NaN`. The code does not crash; your summary just becomes `nan` and any plot
of it goes blank. Decide whether missing values should be dropped (`nan*`
functions) or are themselves a red flag to investigate -- do not let them pass
silently.
:::

:::{admonition} Advanced tips
:class: tip
NumPy 2.0 (2024) removed several long-deprecated aliases. If you see old code break:

* `np.float`, `np.int`, `np.bool` are gone -- use the built-ins `float`, `int`,
  `bool` (or `np.float64` etc.).
* `np.NaN` is now `np.nan`.
* Prefer `rng = np.random.default_rng()` over the legacy `np.random.*` functions;
  it is faster, statistically better, and makes seeding explicit and local.
:::